# Stage 2: MNIST Voting Ensemble

Project goal: use MNIST to compare individual classifiers with hard and soft voting ensembles. The notebook measures accuracy, macro F1, training time, and prediction time, then saves a results table and comparison figure when run.

Stage 1 remains a breast cancer sanity-check baseline. Stage 2 moves to MNIST, which is closer to the voting-ensemble exercise in the book and gives the models a larger, multiclass problem where ensemble behavior is easier to observe.

## Why MNIST for ensemble learning?

The breast cancer dataset is useful for checking that our project code works, but it is small and binary. MNIST has many more observations, 10 classes, and high-dimensional image features. That makes it more suitable for comparing individual classifiers with voting ensembles.

A voting ensemble only helps when its members are both relevant and diverse. If every classifier makes the same mistakes, voting will not add much. If the classifiers learn different useful patterns, voting can reduce variance and sometimes improve accuracy.

In [ ]:
from pathlib import Path
import sys
import time

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.base import clone
from sklearn.datasets import fetch_openml
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

## Data split

Fast mode is the default so the notebook is practical on a laptop. To use the full book-style split, set `FAST_MODE = False`.

- Fast mode: 20,000 train, 5,000 validation, 10,000 test
- Full mode: 50,000 train, 10,000 validation, 10,000 test

In [ ]:
FAST_MODE = True
RANDOM_STATE = 42

if FAST_MODE:
    N_TRAIN = 20_000
    N_VALIDATION = 5_000
    N_TEST = 10_000
else:
    N_TRAIN = 50_000
    N_VALIDATION = 10_000
    N_TEST = 10_000

In [ ]:
def load_mnist_splits(n_train, n_validation, n_test, random_state=42):
    """Load MNIST and return train, validation, and test splits."""
    X, y = fetch_openml("mnist_784", version=1, as_frame=False, return_X_y=True)
    X = X.astype("float32") / 255.0
    y = y.astype("int64")

    total_size = n_train + n_validation + n_test
    if total_size < len(X):
        X, _, y, _ = train_test_split(
            X,
            y,
            train_size=total_size,
            random_state=random_state,
            stratify=y,
        )

    validation_and_train_size = n_train + n_validation
    X_train_valid, X_test, y_train_valid, y_test = train_test_split(
        X,
        y,
        train_size=validation_and_train_size,
        test_size=n_test,
        random_state=random_state,
        stratify=y,
    )

    validation_fraction = n_validation / validation_and_train_size
    X_train, X_validation, y_train, y_validation = train_test_split(
        X_train_valid,
        y_train_valid,
        test_size=validation_fraction,
        random_state=random_state,
        stratify=y_train_valid,
    )

    return X_train, X_validation, X_test, y_train, y_validation, y_test

In [ ]:
X_train, X_validation, X_test, y_train, y_validation, y_test = load_mnist_splits(
    N_TRAIN,
    N_VALIDATION,
    N_TEST,
    random_state=RANDOM_STATE,
)

print(f"Train shape:      X={X_train.shape}, y={y_train.shape}")
print(f"Validation shape: X={X_validation.shape}, y={y_validation.shape}")
print(f"Test shape:       X={X_test.shape}, y={y_test.shape}")

## Models

We compare a random forest, extremely randomized trees, a fast SVM-like linear classifier, and logistic regression. The SVM-like model uses `SGDClassifier` with hinge loss because a full kernel SVM would be computationally expensive on MNIST, especially in full mode.

Logistic regression is included because it provides class probabilities, which are useful for soft voting.

In [ ]:
def make_individual_models(random_state=42):
    """Create the individual classifiers for the MNIST voting experiment."""
    return {
        "Random Forest": RandomForestClassifier(
            n_estimators=100,
            n_jobs=-1,
            random_state=random_state,
        ),
        "Extra-Trees": ExtraTreesClassifier(
            n_estimators=100,
            n_jobs=-1,
            random_state=random_state,
        ),
        "Linear SVM-like SGD": Pipeline(
            steps=[
                ("scaler", StandardScaler()),
                (
                    "model",
                    SGDClassifier(
                        loss="hinge",
                        max_iter=1000,
                        tol=1e-3,
                        random_state=random_state,
                    ),
                ),
            ]
        ),
        "Logistic Regression": Pipeline(
            steps=[
                ("scaler", StandardScaler()),
                (
                    "model",
                    LogisticRegression(
                        max_iter=200,
                        n_jobs=-1,
                        random_state=random_state,
                    ),
                ),
            ]
        ),
    }


individual_models = make_individual_models(random_state=RANDOM_STATE)
individual_models

## Evaluation helpers

For each model, we record accuracy, macro F1, training time, validation prediction time, and test prediction time. Macro F1 gives every digit class equal weight, which is helpful for multiclass classification.

In [ ]:
def evaluate_classifier(model_name, model, X_train, y_train, X_validation, y_validation, X_test, y_test):
    """Fit one classifier and evaluate it on validation and test sets."""
    fitted_model = clone(model)

    start = time.perf_counter()
    fitted_model.fit(X_train, y_train)
    training_time = time.perf_counter() - start

    start = time.perf_counter()
    validation_pred = fitted_model.predict(X_validation)
    validation_prediction_time = time.perf_counter() - start

    start = time.perf_counter()
    test_pred = fitted_model.predict(X_test)
    test_prediction_time = time.perf_counter() - start

    row = {
        "model": model_name,
        "accuracy_validation": accuracy_score(y_validation, validation_pred),
        "macro_f1_validation": f1_score(y_validation, validation_pred, average="macro"),
        "accuracy_test": accuracy_score(y_test, test_pred),
        "macro_f1_test": f1_score(y_test, test_pred, average="macro"),
        "training_time_seconds": training_time,
        "validation_prediction_time_seconds": validation_prediction_time,
        "test_prediction_time_seconds": test_prediction_time,
    }

    return row, fitted_model

## Train individual classifiers

Each model is fit independently. The validation set helps us compare models before looking at the final test scores.

In [ ]:
rows = []
fitted_individual_models = {}

for model_name, model in individual_models.items():
    print(f"Training {model_name}...")
    row, fitted_model = evaluate_classifier(
        model_name,
        model,
        X_train,
        y_train,
        X_validation,
        y_validation,
        X_test,
        y_test,
    )
    rows.append(row)
    fitted_individual_models[model_name] = fitted_model

individual_results = pd.DataFrame(rows).sort_values(
    by="accuracy_validation",
    ascending=False,
).reset_index(drop=True)

individual_results

## Hard voting versus soft voting

Hard voting predicts the class that receives the most votes from the member classifiers. Soft voting averages predicted class probabilities and picks the class with the highest average probability.

Soft voting can work well when the probability estimates are meaningful, but it can only use models that implement `predict_proba`. The SVM-like SGD model does not provide probabilities in this configuration, so it is included in hard voting but not soft voting.

In [ ]:
hard_voting = VotingClassifier(
    estimators=[
        ("random_forest", individual_models["Random Forest"]),
        ("extra_trees", individual_models["Extra-Trees"]),
        ("linear_svm_like_sgd", individual_models["Linear SVM-like SGD"]),
        ("logistic_regression", individual_models["Logistic Regression"]),
    ],
    voting="hard",
)

soft_voting = VotingClassifier(
    estimators=[
        ("random_forest", individual_models["Random Forest"]),
        ("extra_trees", individual_models["Extra-Trees"]),
        ("logistic_regression", individual_models["Logistic Regression"]),
    ],
    voting="soft",
)

for model_name, model in {"Hard Voting": hard_voting, "Soft Voting": soft_voting}.items():
    print(f"Training {model_name}...")
    row, fitted_model = evaluate_classifier(
        model_name,
        model,
        X_train,
        y_train,
        X_validation,
        y_validation,
        X_test,
        y_test,
    )
    rows.append(row)
    fitted_individual_models[model_name] = fitted_model

## Compare results

The table compares individual classifiers, hard voting, and soft voting. Remember that the best ensemble is not guaranteed to beat every individual model. Ensemble performance depends on the relevance of the base learners and the diversity of their errors.

In [ ]:
results = pd.DataFrame(rows).sort_values(
    by="accuracy_validation",
    ascending=False,
).reset_index(drop=True)

results

## Save outputs

Running this cell writes the requested CSV table and comparison figure. The `results/` directory is created if it does not already exist.

In [ ]:
tables_dir = Path("results/tables")
figures_dir = Path("results/figures")
tables_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

csv_path = tables_dir / "stage2_mnist_voting_results.csv"
figure_path = figures_dir / "stage2_mnist_model_comparison.png"

results.to_csv(csv_path, index=False)

plot_data = results.set_index("model")
fig, axes = plt.subplots(ncols=2, figsize=(14, 5), sharey=True)

plot_data[["accuracy_validation", "accuracy_test"]].plot(
    kind="bar",
    ax=axes[0],
    title="Accuracy",
)
plot_data[["macro_f1_validation", "macro_f1_test"]].plot(
    kind="bar",
    ax=axes[1],
    title="Macro F1",
)

for ax in axes:
    ax.set_xlabel("")
    ax.set_ylim(0.0, 1.0)
    ax.tick_params(axis="x", labelrotation=45)
    ax.legend(loc="lower right")

fig.suptitle("MNIST voting ensemble comparison")
fig.tight_layout()
fig.savefig(figure_path, dpi=150, bbox_inches="tight")

print(f"Saved table to {csv_path}")
print(f"Saved figure to {figure_path}")

## Learning checkpoint

1. Which individual classifier has the best validation accuracy?
2. Does hard voting improve over the best individual classifier?
3. Does soft voting improve over hard voting?
4. Which model is most expensive to train? Is the extra cost worth it here?
5. Do the ensemble results suggest enough relevance and diversity among the base learners?